# M10 · Build BI dashboard views

**Goal:** create (or replace) the three BI dashboard views that the front-end consumes:
1. `marts.v_executive_dashboard` (sql/33) — tenant-month KPIs + MoM deltas + 3-mo rolling
2. `marts.v_operational_dashboard` (sql/34) — daily KPIs + per-100km ratios + 7d rolling
3. `marts.v_maintenance_dashboard` (sql/35) — vehicle-month maintenance leaderboard

Views are stateless — re-running this notebook is a `CREATE OR REPLACE` and never moves data.

In [1]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

from accent_fleet.db import get_engine, run_sql_file, transaction
from sqlalchemy import text
import pandas as pd

## 2. Inputs

In [2]:
VIEWS = [
    '33_v_executive_dashboard.sql',
    '34_v_operational_dashboard.sql',
    '35_v_maintenance_dashboard.sql',
]
VIEWS

['33_v_executive_dashboard.sql',
 '34_v_operational_dashboard.sql',
 '35_v_maintenance_dashboard.sql']

## 3. Execute

In [3]:
with transaction() as conn:
    for f in VIEWS:
        run_sql_file(conn, f)
        print(f'ok: {f}')
print('all BI views (re)created.')

ok: 33_v_executive_dashboard.sql
ok: 34_v_operational_dashboard.sql
ok: 35_v_maintenance_dashboard.sql
all BI views (re)created.


## 4. Inspect

In [4]:
with get_engine().connect() as conn:
    exec_dash = pd.read_sql(text('SELECT * FROM marts.v_executive_dashboard ORDER BY tenant_id, year_month DESC LIMIT 5'), conn)
    op_dash   = pd.read_sql(text('SELECT * FROM marts.v_operational_dashboard ORDER BY fleet_date DESC LIMIT 5'), conn)
    maint_dash = pd.read_sql(text('SELECT * FROM marts.v_maintenance_dashboard ORDER BY year_month DESC, total_cost DESC LIMIT 5'), conn)
display(exec_dash); display(op_dash); display(maint_dash)

,tenant_id,year_month,active_vehicles,active_devices,total_trips,total_distance_km,total_maintenance_cost,total_fuel_cost,total_operating_cost,cost_per_km,total_alerts,panic_alerts,total_overspeed,total_harsh_events,distance_km_mom_delta,operating_cost_mom_delta,alerts_mom_delta,distance_km_3mo_avg,cost_per_km_3mo_avg
0,235,2026-04,134,126,15153,202029.395509,0.0,0.0,0.0,0.0,7,0,1422,26109,-271587.066354,0.0,-78,364426.426265,0.0
1,235,2026-03,134,125,33031,473616.461863,0.0,0.0,0.0,0.0,85,0,2991,69781,55983.040438,0.0,12,442894.744186,0.0
2,235,2026-02,137,125,32665,417633.421424,0.0,0.0,0.0,0.0,73,0,3495,57477,-19800.927847,0.0,-28,469395.710488,0.0
3,235,2026-01,139,123,34999,437434.349271,0.0,0.0,0.0,0.0,101,0,3597,50341,-115685.011498,0.0,25,506914.750712,0.0
4,235,2025-12,140,131,43492,553119.360769,0.0,0.0,0.0,0.0,76,0,4788,59189,22928.818673,0.0,-8,577179.811212,0.0


,tenant_id,fleet_date,active_devices,total_trips,total_distance_km,total_driving_hours,total_stops,total_alerts,speed_alerts,geofence_alerts,...,harsh_brake_events,harsh_accel_events,harsh_corner_events,total_harsh_events,alerts_per_100km,overspeed_per_100km,harsh_events_per_100km,panic_share_pct,distance_km_7d_avg,alerts_7d_avg
0,1787,2026-04-10,1,1,0.070674,0.260833,1,0,0,0,...,0,0,0,0,0.000000,0.000000,0.000000,NaN,7853.557837,10.428571
1,238,2026-04-10,44,338,4388.604381,113.363611,364,4,0,0,...,241,833,591,1665,0.091145,0.136718,37.939168,0.0,7066.984747,6.000000
2,235,2026-04-10,120,1290,12975.322142,381.884167,1430,2,0,0,...,503,1246,1018,2767,0.015414,1.248524,21.325097,0.0,19303.521010,1.000000
3,264,2026-04-10,47,408,8161.560164,177.028056,442,384,0,0,...,42,26,229,297,4.704983,7.792628,3.639010,0.0,13437.965327,596.714286
4,7486,2026-04-10,0,0,0.000000,NaN,5,0,0,0,...,1,1,3,5,NaN,NaN,NaN,NaN,0.000000,55.000000


,tenant_id,vehicle_id,matricule,vehicle_mark,vehicle_class,year_month,total_distance_km,active_days,maintenance_events,offense_events,...,maintenance_labor_total,reparation_amount_total,fuel_cost_total,total_cost,maintenance_share_pct,cost_per_km,fuel_l_per_100km,avg_repair_hours,max_repair_hours,cost_rank_in_tenant
0,235,33,6489TU207 Renault,425138,unknown,2026-04,1206.325159,10,0,0,...,0.0,0.0,0.0,0.0,None,0.0,None,None,None,1
1,235,34,1210TU193 mohamed,Scania,heavy,2026-04,1712.943998,8,0,0,...,0.0,0.0,0.0,0.0,None,0.0,None,None,None,1
2,235,29,8642TU200 Scania,8642tu200 Scania,unknown,2026-04,133.417400,2,0,0,...,0.0,0.0,0.0,0.0,None,0.0,None,None,None,1
3,235,32,Amer 1970TU193,425115,unknown,2026-04,1703.796076,7,0,0,...,0.0,0.0,0.0,0.0,None,0.0,None,None,None,1
4,235,35,345TU207 Iveco ezidine,425139,unknown,2026-04,762.453619,9,0,0,...,0.0,0.0,0.0,0.0,None,0.0,None,None,None,1
